# מפגש 12 — מחברת הדגמה למורה
המטרה: להבין יעד ומאפיינים, חלוקה משותפת והשלמת חסרים מאימון בלבד. **לא מאמנים מודל**. התלמידים יפתרו בהמשך משימה עצמאית על תכונות שונות ו־seed שונה.

## דוגמה ידנית: Train ולא Test מלמדים את הממוצע
Train=[10,20,NaN], Test=[1000,NaN]. צפו mean_train=15 ו־Train חדש [10,20,15], Test חדש [1000,15]. מאגר זה מומצא.

In [ ]:
import pandas as pd
import numpy as np
train=pd.DataFrame({'minutes':[10.,20.,None]})
test=pd.DataFrame({'minutes':[1000.,None]})
mean_train=train['minutes'].mean()
train_ready=train.fillna({'minutes':mean_train})
test_ready=test.fillna({'minutes':mean_train})
print('Train mean:',mean_train)
print('Train filled:',train_ready['minutes'].tolist())
print('Test filled:',test_ready['minutes'].tolist())
assert mean_train==15
assert train_ready['minutes'].tolist()==[10,20,15]
assert test_ready['minutes'].tolist()==[1000,15]

## למה אי אפשר ללמוד את הממוצע מכל הדוגמאות?
בנתונים הידועים של Train ו־Test יחד: (10+20+1000)/3≈343.33. זו דרך פסולה להכנת נתוני אימון. הריצה הבאה לצורכי *הדגמה בלבד*.

In [ ]:
wrong_mean=pd.concat([train,test],ignore_index=True)['minutes'].mean()
print('Wrong all-data mean:',wrong_mean)
assert np.isclose(wrong_mean,1030/3)
assert wrong_mean!=mean_train

## מעבר למאגר מדעי — מדידות Penguins
מפת המקור: 344×7; species יעד, bill_length_mm ו־bill_depth_mm הם מדידות ההדגמה. אין למחוק שורה בגלל sex חסר.

In [ ]:
from sklearn.model_selection import train_test_split
url='https://raw.githubusercontent.com/samiabo75/Data-Science-ML-Course/main/Datasets/penguins.csv'
df=pd.read_csv(url)
original=df.copy(deep=True)
print('Source shape:',df.shape)
print('Missing by field:')
print(df.isna().sum())
assert df.shape==(344,7)
assert df['species'].isna().sum()==0
assert df['sex'].isna().sum()==11

## מזהים X ו־y תוך שמירה על התאמה
בדוגמת המורה נבחר bill_length_mm ו־bill_depth_mm, **לא** flipper_length_mm ו־body_mass_g שהם תרגיל התלמיד.

In [ ]:
model_df=df.dropna(subset=['species']).copy()
X=model_df[['bill_length_mm','bill_depth_mm']]
y=model_df['species']
print('X shape:',X.shape,'y shape:',y.shape)
print('Missing X:')
print(X.isna().sum())
assert X.shape==(344,2)
assert y.shape==(344,)
assert 'species' not in X.columns
assert X.index.equals(y.index)

## חלוקה משותפת. Seed 42 בהדגמה
הערה: מטלת התלמיד משתמשת ב־seed 17 ובשדות אחרים. stratify שומרת בקירוב התפלגות מינים, ולא מסירה חסרים.

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.25,random_state=42,stratify=y)
print('Train and Test:',X_train.shape,X_test.shape)
print('Missing Train:',X_train.isna().sum().to_dict())
print('Missing Test:',X_test.isna().sum().to_dict())
assert len(X_train)==258 and len(X_test)==86
assert X_train.index.equals(y_train.index)
assert X_test.index.equals(y_test.index)
assert set(X_train.index).isdisjoint(set(X_test.index))

## ערכי השלמה: ללמוד רק מ־Train ולהפעיל בשתי הקבוצות
גם אם כל החסרים נפלו ב־Train או רק ב־Test, המדיניות אינה משתנה.

In [ ]:
training_means=X_train.mean()
X_train_ready=X_train.fillna(training_means)
X_test_ready=X_test.fillna(training_means)
print('Train means:',training_means.to_dict())
print('Missing after:',X_train_ready.isna().sum().to_dict(),X_test_ready.isna().sum().to_dict())
assert not X_train_ready.isna().any().any()
assert not X_test_ready.isna().any().any()
assert df.equals(original)
assert X_train.index.equals(X_train_ready.index)
assert X_test.index.equals(X_test_ready.index)

## כרטיס יציאה
האם נמדד Accuracy? לא. האם למדנו ממוצע מן Test? לא. מה הבעיה במילוי species חסר בלי תווית אמת? כיצד מונעים דליפה גם בזמן אימות עתידי?